### Linda Zier
### ST 554
### Project 2

**Introduction**

This project has two parts.  In the first part I have created a data quality class in pyspark. This class is a python class that wraps an SQL DataFrame and provides functionality for cleaning and checking data.

In the second part of the project I analyze data using spark SQL style dataframes and pandas-on-spark data frames.



In [161]:

from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from functools import reduce
from pyspark.sql.types import *
import pandas as pd

from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

In [162]:
import sys
sys.path.insert(0,'.')

import importlib
import Spark_class
importlib.reload(Spark_class)

from Spark_class import SparkDataCheck

I've created a data folder on my github repository and stored the air quality data there.  This project utilizes the air quality data set available at the UCI Machine Learning Repository (https://archive.ics.uci.edu/dataset/360/air+quality). This data is time series data (data recorded over time) for air quality measurements of pollutants in an Italian city. The dataset consists of 9,358 hourly-averaged observations collected from five metal oxide chemical sensors integrated into an air quality multisensor device. The device was deployed at road level in a heavily polluted urban area in Italy, and data were gathered over a one-year period from March 2004 through February 2005 (De Vito, 2008).

The pollutants measured, which are "true" (gold standard) measurements, are the following:

- CO(GT) (CO concentration)
- C6H6(GT) (Benzene concentration)
- NMHC(GT) (Non Metanic HydroCarbon concentration)
- NOx(GT) (NOx concentration)
- NO2(GT) (NO2 concentration)
- T (temperature)
- RH (Relative humidity)
- AH (Absolute Humidity)

The data has missing values indicated by an entry of -200 so I started by replacing those with None to give NULL values.

In [163]:

# Loading the air data using the from_csv method
air = Spark_class.SparkDataCheck.from_csv(spark, "data/air.csv")
   
air.df.printSchema(5)

#get numeric columns
numeric_types = ['int', 'bigint', 'float', 'double', 'decimal', 'long', 'short']
#columns_to_clean = [c for c, dtype in air.df.dytpes

#cleaning up : replacing the -200 values with NULL
for c, dtype in air.df.dtypes:
    if dtype in numeric_types:
        air.df=air.df.withColumn(c, F.when(F.col(f'`{c}`')== -200, None).otherwise(F.col(f'`{c}`')))
    
air.df.show(5)

root
 |-- _c0: integer (nullable = true)
 |-- Date: string (nullable = true)
 |-- Time: timestamp (nullable = true)
 |-- CO(GT): double (nullable = true)
 |-- PT08.S1(CO): integer (nullable = true)
 |-- NMHC(GT): integer (nullable = true)
 |-- C6H6(GT): double (nullable = true)
 |-- PT08.S2(NMHC): integer (nullable = true)
 |-- NOx(GT): integer (nullable = true)
 |-- PT08.S3(NOx): integer (nullable = true)
 |-- NO2(GT): integer (nullable = true)
 |-- PT08.S4(NO2): integer (nullable = true)
 |-- PT08.S5(O3): integer (nullable = true)
 |-- T: double (nullable = true)
 |-- RH: double (nullable = true)
 |-- AH: double (nullable = true)

+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+
|_c0|     Date|               Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|
+---+---------+-------------------+------+-

26/03/22 21:14:07 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-ljzier@ncsu.edu/ST-554-project2-repo/data/air.csv


In order to test the string methods, I needed to add a few more columns with string variables.  It made sense to quantify temperature and relative humidity. I also needed to ensure the NULL values didn't end up in one of the other categories to falsely record the level.

In [184]:
# making some string columns that hopefully make sense

air.df = air.df.withColumn('T_level',F.when(F.col('T') > 20, 'high').
                           when(F.col('T') > 10, 'medium').when(F.col('T') >= 0, 'low').otherwise('NULL'))

air.df = air.df.withColumn('RH_level',F.when(F.col('RH') > 60,'high').
                           when(F.col('RH')>30, 'medium').when(F.col('RH') >= 0, 'low').otherwise('NULL'))

air.df.show()

#checking to make sure the NULL values don't end up in one of the other categories to falsely record the level
result.df.filter(F.col('null_check') == True).show(10)

+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+-------+--------+---------+----------+---------+
|_c0|     Date|               Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|T_level|RH_level|num_check|null_check|str_check|
+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+-------+--------+---------+----------+---------+
|  0|3/10/2004|2026-03-22 18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578| medium|  medium|     true|     false|     true|
|  1|3/10/2004|2026-03-22 19:00:00|   2.0|       1292|     112|     9.4|          955|    103|        1174|     92|        1559|        972|13.3|47.7|0.7255| medium|  medium|  

26/03/22 21:22:08 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-ljzier@ncsu.edu/ST-554-project2-repo/data/air.csv
26/03/22 21:22:08 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-ljzier@ncsu.edu/ST-554-project2-repo/data/air.csv


### Examples

- Here is an example of using my methods on the air quality data to check between 2 values.

In [165]:
result=air.check_col_range(column="T", lower=10.0, upper=11.1)
result.df.show(10)

+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+-------+--------+---------+
|_c0|     Date|               Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|T_level|RH_level|num_check|
+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+-------+--------+---------+
|  0|3/10/2004|2026-03-22 18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578| medium|  medium|    false|
|  1|3/10/2004|2026-03-22 19:00:00|   2.0|       1292|     112|     9.4|          955|    103|        1174|     92|        1559|        972|13.3|47.7|0.7255| medium|  medium|    false|
|  2|3/10/2004|2026-03-22 20:00:00|   2.2|       1402|      88|     9.0|   

26/03/22 21:14:07 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-ljzier@ncsu.edu/ST-554-project2-repo/data/air.csv


- Now I'm showing what happens when you don't have at least a lower or an upper.

In [166]:
result=air.check_col_range(column="T")
#result.df.show(10)

Need a lower and/or an upper range for T


- Here I've supplied only an upper range in one case and a lower case in another.

In [167]:
air.check_col_range(column="CO(GT)", upper=2.0)
air.df.show(10)

air.check_col_range(column="NOx(GT)", lower=100)
air.df.show(10)

+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+-------+--------+---------+
|_c0|     Date|               Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|T_level|RH_level|num_check|
+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+-------+--------+---------+
|  0|3/10/2004|2026-03-22 18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578| medium|  medium|    false|
|  1|3/10/2004|2026-03-22 19:00:00|   2.0|       1292|     112|     9.4|          955|    103|        1174|     92|        1559|        972|13.3|47.7|0.7255| medium|  medium|     true|
|  2|3/10/2004|2026-03-22 20:00:00|   2.2|       1402|      88|     9.0|   

26/03/22 21:14:07 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-ljzier@ncsu.edu/ST-554-project2-repo/data/air.csv
26/03/22 21:14:07 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-ljzier@ncsu.edu/ST-554-project2-repo/data/air.csv


- Here I've tested something that wasn't one of the number types we specified.

In [185]:
air.check_col_range(column="T_level", lower=0)


Column is not numeric


In [186]:
# this was a visual for me to see what types I had to work with.
#air.df.dtypes

**Now I'm going to give some examples of checking for null values**
- First I am checking for NULL values in the T (temperature) column. I'm also filtering on just Null in that column so it's easy to check.

In [170]:
#air.df.toPandas().isnull().sum()

# check for null in T
result=air.check_null(column="T")
result.df.show()

result.df.filter(F.col('null_check') == True).show(10)

26/03/22 21:14:08 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-ljzier@ncsu.edu/ST-554-project2-repo/data/air.csv


+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+-------+--------+---------+----------+
|_c0|     Date|               Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|T_level|RH_level|num_check|null_check|
+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+-------+--------+---------+----------+
|  0|3/10/2004|2026-03-22 18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578| medium|  medium|     true|     false|
|  1|3/10/2004|2026-03-22 19:00:00|   2.0|       1292|     112|     9.4|          955|    103|        1174|     92|        1559|        972|13.3|47.7|0.7255| medium|  medium|     true|     false|
|  2|3/10/2004|2026-

26/03/22 21:14:08 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-ljzier@ncsu.edu/ST-554-project2-repo/data/air.csv


In [171]:
# I used this to see counts 
# air.df.toPandas().isnull().sum()

- Below I am checking for NULL values in the RH (relative humidity) column. I also filter on NULL so it's easy to verify.

In [172]:
# check for null in RH
result=air.check_null(column="RH")
result.df.show()

result.df.filter(F.col('null_check') == True).show(10)

+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+-------+--------+---------+----------+
|_c0|     Date|               Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|T_level|RH_level|num_check|null_check|
+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+-------+--------+---------+----------+
|  0|3/10/2004|2026-03-22 18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578| medium|  medium|     true|     false|
|  1|3/10/2004|2026-03-22 19:00:00|   2.0|       1292|     112|     9.4|          955|    103|        1174|     92|        1559|        972|13.3|47.7|0.7255| medium|  medium|     true|     false|
|  2|3/10/2004|2026-

26/03/22 21:14:08 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-ljzier@ncsu.edu/ST-554-project2-repo/data/air.csv
26/03/22 21:14:08 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-ljzier@ncsu.edu/ST-554-project2-repo/data/air.csv


- Here I'm checking for null values in the date column.  There are no missing values so this was a good way to verify that it was working.

In [173]:
# check for null in Date
result=air.check_null(column="Date")
result.df.show()

result.df.filter(F.col('null_check') == True).show(10)

26/03/22 21:14:08 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-ljzier@ncsu.edu/ST-554-project2-repo/data/air.csv


+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+-------+--------+---------+----------+
|_c0|     Date|               Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|T_level|RH_level|num_check|null_check|
+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+-------+--------+---------+----------+
|  0|3/10/2004|2026-03-22 18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578| medium|  medium|     true|     false|
|  1|3/10/2004|2026-03-22 19:00:00|   2.0|       1292|     112|     9.4|          955|    103|        1174|     92|        1559|        972|13.3|47.7|0.7255| medium|  medium|     true|     false|
|  2|3/10/2004|2026-

26/03/22 21:14:08 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-ljzier@ncsu.edu/ST-554-project2-repo/data/air.csv


- Here I'm checking for null values in the Benzene concentration(C6H6(GT)) column. 

In [174]:
# check for null in C6H6(GT)
result=air.check_null(column="C6H6(GT)")
result.df.show()

result.df.filter(F.col('null_check') == True).show(10)

+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+-------+--------+---------+----------+
|_c0|     Date|               Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|T_level|RH_level|num_check|null_check|
+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+-------+--------+---------+----------+
|  0|3/10/2004|2026-03-22 18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578| medium|  medium|     true|     false|
|  1|3/10/2004|2026-03-22 19:00:00|   2.0|       1292|     112|     9.4|          955|    103|        1174|     92|        1559|        972|13.3|47.7|0.7255| medium|  medium|     true|     false|
|  2|3/10/2004|2026-

26/03/22 21:14:08 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-ljzier@ncsu.edu/ST-554-project2-repo/data/air.csv
26/03/22 21:14:08 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-ljzier@ncsu.edu/ST-554-project2-repo/data/air.csv


- Now I can check if each value in a string column falls within a user specified set of levels and returns the dataframe with an appended column of Boolean values. Below I've checked various string column levels.

In [175]:
print('\n\n Temp level = low')
air.check_string('T_level', 'low')
air.df.show()

print('\n\n RH level = high')
air.check_string('RH_level', 'high')
air.df.show() 

print('\n\n Temp level = medium')
air.check_string('T_level', 'medium')
air.df.show()

# you can see that these values are the same as previous because RH is not a string
air.check_string('RH', 'medium')
air.df.show()




 Temp level = low


26/03/22 21:14:08 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-ljzier@ncsu.edu/ST-554-project2-repo/data/air.csv


+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+-------+--------+---------+----------+---------+
|_c0|     Date|               Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|T_level|RH_level|num_check|null_check|str_check|
+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+-------+--------+---------+----------+---------+
|  0|3/10/2004|2026-03-22 18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578| medium|  medium|     true|     false|    false|
|  1|3/10/2004|2026-03-22 19:00:00|   2.0|       1292|     112|     9.4|          955|    103|        1174|     92|        1559|        972|13.3|47.7|0.7255| medium|  medium|  

26/03/22 21:14:08 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-ljzier@ncsu.edu/ST-554-project2-repo/data/air.csv


+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+-------+--------+---------+----------+---------+
|_c0|     Date|               Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|T_level|RH_level|num_check|null_check|str_check|
+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+-------+--------+---------+----------+---------+
|  0|3/10/2004|2026-03-22 18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578| medium|  medium|     true|     false|    false|
|  1|3/10/2004|2026-03-22 19:00:00|   2.0|       1292|     112|     9.4|          955|    103|        1174|     92|        1559|        972|13.3|47.7|0.7255| medium|  medium|  

26/03/22 21:14:08 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-ljzier@ncsu.edu/ST-554-project2-repo/data/air.csv
26/03/22 21:14:08 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-ljzier@ncsu.edu/ST-554-project2-repo/data/air.csv


**Summarization method examples**

Below I've given some examples of the summarization methods.

- This summarizes all the numeric columns since no column was entered in the method call.

In [176]:
air.col_minmax()

26/03/22 21:14:08 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-ljzier@ncsu.edu/ST-554-project2-repo/data/air.csv


,col_name,min,max
0,_c0,0.0000,9356.000
1,CO(GT),0.1000,11.900
2,PT08.S1(CO),647.0000,2040.000
3,NMHC(GT),7.0000,1189.000
4,C6H6(GT),0.1000,63.700
5,PT08.S2(NMHC),383.0000,2214.000
6,NOx(GT),2.0000,1479.000
7,PT08.S3(NOx),322.0000,2683.000
8,NO2(GT),2.0000,340.000
9,PT08.S4(NO2),551.0000,2775.000


- Now I summarize a few diffent numeric columns to show functionality. This includes 2 examples of grouped levels

In [177]:
air.col_minmax("CO(GT)")

,col_name,min,max
0,CO(GT),0.1,11.9


In [178]:
air.col_minmax("C6H6(GT)", "T_level")

,col_name,T_level,min,max
0,C6H6(GT),low,0.1,63.7
1,C6H6(GT),high,1.0,52.1
2,C6H6(GT),medium,0.2,50.8


In [179]:
air.col_minmax("C6H6(GT)", "RH_level")

,col_name,RH_level,min,max
0,C6H6(GT),low,0.9,38.4
1,C6H6(GT),high,0.5,52.1
2,C6H6(GT),medium,0.1,63.7


In this case I call a non-numeric column

In [180]:
air.col_minmax("T_level")
               

Column T_level is not numeric


Finally, I will report the counts associated with one or two string columns.
- Below are the temporature counts for each category.

In [187]:
air.str_count('T_level')

,T_level,count
0,low,1683
1,high,3711
2,medium,3584
3,NULL,379


- Below are the relative humidity counts for each category.

In [188]:
air.str_count('RH_level')

,RH_level,count
0,low,1441
1,high,2633
2,medium,4917
3,NULL,366


In [193]:
result=air.str_count('T_level', 'RH_level')
result.sort_values(['T_level', 'RH_level'])

,T_level,RH_level,count
6,NULL,NULL,366
4,NULL,medium,13
1,high,high,375
8,high,low,1186
5,high,medium,2150
0,low,high,771
7,low,low,33
10,low,medium,879
3,medium,high,1487
9,medium,low,222


In [195]:
#result['T_level'] = pd.Categorical(result['T_level'], categories=['low', 'medium', 'high'], ordered=True)
#result['RH_level'] = pd.Categorical(result['RH_level'], categories=['low', 'medium', 'high'], ordered=True)
#result.sort_values(['T_level', 'RH_level'])